### 1. Audyt plików i folderów (Sprawdzanie co się wygenerowało)
Ta funkcja pokaże Ci dokładnie, ile plików masz w poszczególnych podfolderach, jakie mają rozszerzenia i ile łącznie zajmują miejsca.

In [1]:
import os
import pandas as pd
from pathlib import Path

def audit_directory(path):
    """Generuje szczegółowy raport o zawartości folderu."""
    if not os.path.exists(path):
        print(f"❌ Folder nie istnieje: {path}")
        return

    print(f"🔍 Rozpoczynam audyt: {path}")
    stats = []
    for root, dirs, files in os.walk(path):
        for f in files:
            p = Path(os.path.join(root, f))
            stats.append({
                'folder': os.path.basename(root),
                'ext': p.suffix.lower(),
                'size_mb': p.stat().st_size / (1024 * 1024)
            })
    
    if not stats:
        print("⚠️ Folder jest pusty!")
        return
        
    df = pd.DataFrame(stats)
    summary = df.groupby(['folder', 'ext']).agg(
        ilosc=('size_mb', 'count'),
        rozmiar_total_mb=('size_mb', 'sum')
    ).round(2)
    
    print("\n📊 Podsumowanie zawartości:")
    print(summary)
    print(f"\n📦 Całkowity rozmiar wszystkich plików: {df['size_mb'].sum():.2f} MB")

# Użycie (zmień ścieżkę na swoją, np. /content/1K_out):
audit_directory('/content/1K_out')

🔍 Rozpoczynam audyt: /content/1K_out

📊 Podsumowanie zawartości:
              ilosc  rozmiar_total_mb
folder ext                           
1K_out .json      1               0.0

📦 Całkowity rozmiar wszystkich plików: 0.00 MB


### 2. Weryfikacja Manifestu (Czy JSONL nie jest uszkodzony)
Sprawdza, czy plik manifestu nie został "ucięty" w połowie podczas zapisu i czy każda linia jest poprawnym obiektem JSON.

In [2]:
import json

def check_manifest_health(manifest_path):
    """Sprawdza spójność pliku manifest.jsonl."""
    if not os.path.exists(manifest_path):
        print(f"❌ Nie znaleziono manifestu: {manifest_path}")
        return

    print(f"📄 Analiza manifestu: {manifest_path}")
    count = 0
    errors = 0
    with open(manifest_path, 'r') as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line: continue
            try:
                json.loads(line)
                count += 1
            except json.JSONDecodeError:
                print(f"  ❌ BŁĄD w linii {i}: Niepoprawny format JSON")
                errors += 1
                
    print(f"\n✅ Poprawnych wpisów: {count}")
    if errors > 0:
        print(f"⚠️ Znaleziono {errors} uszkodzonych linii!")
    else:
        print("✨ Manifest jest w 100% zdrowy.")

# Użycie:
check_manifest_health('/content/1K_out/manifest.jsonl')

❌ Nie znaleziono manifestu: /content/1K_out/manifest.jsonl


### 3. Sprawdzanie wolnego miejsca w Colab
Zawsze warto sprawdzić przed startem, czy lokalny dysk `/content` udźwignie 1000 zdjęć.

In [3]:
import shutil

def check_colab_disk():
    """Pokazuje wolne miejsce na dysku maszyny Colab."""
    usage = shutil.disk_usage("/content")
    free_gb = usage.free / (1024**3)
    total_gb = usage.total / (1024**3)
    percent_free = (free_gb / total_gb) * 100
    
    print(f"💾 Dysk lokalny (/content):")
    print(f"   Wolne: {free_gb:.2f} GB")
    print(f"   Razem: {total_gb:.2f} GB")
    print(f"   Dostępne: {percent_free:.1f}%")
    
    if free_gb < 5:
        print("⚠️ UWAGA: Mało miejsca na dysku! Pipeline może się przerwać.")

check_colab_disk()

💾 Dysk lokalny (/content):
   Wolne: 86.76 GB
   Razem: 107.72 GB
   Dostępne: 80.5%
